In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal, Annotated
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

In [2]:
load_dotenv()  # Load environment variables from .env file

True

In [3]:
generator_llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash")
evaluator_llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")
optimizer_llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash")

In [ ]:
#state definition
class TweetState(TypedDict):
    topic: str
    iteration: int
    tone: Literal["funny", "serious", "informative"]
    content: str
    evaluation: Literal["approved", "needs improvement"]
    max_tries: int
    feedback: Annotated[str, "Feedback from the evaluator on how to improve the tweet."]

In [6]:
graph = StateGraph(TweetState)

In [7]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_core.messages import SystemMessage, HumanMessage

def generate_tweet(state: TweetState):
    topic = state["topic"]
    tone = state["tone"]
    parser = StrOutputParser()
    max_tries = state['max_tries']
    iteration = state['iteration']

    messages = [
        SystemMessage(content="You are an expert content creator proficient in creating engaging social media content with human-like tone and style."),
        HumanMessage(content="""Generate a tweet about the topic provided, following the rules specified.
        Write a short, original tweet in a {tone} tone about {topic}. 

        Rules:
        - Do NOT copy or reproduce any existing content.
        - Max 300 characters per tweet.
        - Use simple, clear language.
        - This is try number {iteration} of {max_tries}.
        
        """),
    ]

    chain = generator_llm | parser
    tweet = chain.invoke(messages)
    return {"tweet" : tweet, "iteration" : iteration+1}

